# Tutorial 16: Gene & Protein Annotation + Weighted Embeddings

This notebook covers three topics:

1. **Gene annotation** -- pathways, tissue expression, PPI, diseases
2. **Protein annotation** -- UniProt functional metadata
3. **Weighted protein embeddings** -- TPM-weighted isoform averaging,
   annotation-weighted residue pooling, expression-context concatenation

In [ ]:
import logging
logging.basicConfig(level=logging.WARNING)

## Part 1: Gene Annotation

The ``GeneAnnotator`` fetches metadata from MyGene.info, GTEx,
STRING-DB, Open Targets, and the GWAS Catalog.

In [ ]:
from embpy.resources import GeneAnnotator

ga = GeneAnnotator()

# Pathway annotations
pathways = ga.get_pathways("TP53")
print(f"Reactome pathways: {len(pathways['reactome'])}")
print(f"KEGG pathways: {len(pathways['kegg'])}")
for p in pathways['reactome'][:3]:
    print(f"  {p['id']}: {p['name']}")

In [ ]:
# Tissue expression from GTEx
tissues = ga.get_tissue_expression("TP53")
print(f"Tissues with expression data: {len(tissues)}")
for t in tissues[:5]:
    print(f"  {t['tissue_name']:30s} TPM={t['median_tpm']:.1f}")

In [ ]:
# PPI partners from STRING-DB
partners = ga.get_protein_interactions("TP53", n_partners=5)
for p in partners:
    print(f"  {p['partner']:10s} score={p['combined_score']}")

In [ ]:
# Disease associations from Open Targets
diseases = ga.get_disease_associations("TP53", top_n=5)
for d in diseases:
    print(f"  {d['disease_name']:30s} score={d['score']:.3f}")

In [ ]:
# Full annotation in one call
full = ga.annotate("BRCA1", sources=["pathways", "diseases"])
print(f"Pathways: {sum(len(v) for v in full['pathways'].values())}")
print(f"Diseases: {len(full['disease_associations'])}")

## Part 2: Protein Annotation (UniProt)

The ``ProteinAnnotator`` fetches functional metadata directly
from the UniProt JSON API.

In [ ]:
from embpy.resources import ProteinAnnotator

pa = ProteinAnnotator()

result = pa.annotate("TP53", sources=["metadata", "function", "location", "sites"])

print(f"Protein: {result['metadata']['protein_name']}")
print(f"Reviewed: {result['metadata']['reviewed']}")
print(f"Length: {result['metadata']['sequence_length']} aa")
print(f"\nFunction: {result['function']['function'][0][:100]}...")
print(f"\nSubcellular locations:")
for loc in result['subcellular_location']:
    print(f"  {loc['location']}")
print(f"\nFunctional sites:")
print(f"  Active sites: {len(result['functional_sites']['active_sites'])}")
print(f"  Binding sites: {len(result['functional_sites']['binding_sites'])}")
print(f"  Motifs: {len(result['functional_sites']['motifs'])}")

In [ ]:
# GO terms
result_go = pa.annotate("TP53", sources="go")
go = result_go['go_terms']
print(f"Molecular Function: {len(go['molecular_function'])} terms")
print(f"Biological Process: {len(go['biological_process'])} terms")
print(f"Cellular Component: {len(go['cellular_component'])} terms")
for t in go['molecular_function'][:3]:
    print(f"  {t['id']}: {t['term']}")

In [ ]:
# Isoform-specific annotations
result_iso = pa.annotate("TP53", sources="isoforms")
for iso in result_iso.get('isoform_annotations', []):
    print(f"  {iso['id']}: {iso['name']} ({iso['sequence_status']})")

## Part 3: Weighted Protein Embeddings

The ``WeightedProteinEmbedder`` combines protein language model
embeddings with biological context for perturbation modeling.

In [ ]:
from embpy.embedder import BioEmbedder
from embpy.tl import WeightedProteinEmbedder

embedder = BioEmbedder(device="auto")
wpe = WeightedProteinEmbedder(embedder)

In [ ]:
# Strategy 1: TPM-weighted isoform average
emb_tpm = wpe.tpm_weighted_embedding(
    "TP53", model="esm2_650M",
    tpm_values={"P04637": 45.2, "P04637-2": 12.8, "P04637-3": 2.1},
)
print(f"TPM-weighted shape: {emb_tpm.shape}")

In [ ]:
# Strategy 2: Annotation-weighted residue pooling
emb_ann = wpe.annotation_weighted_embedding(
    "TP53", model="esm2_650M",
    weight_sites=["active_sites", "binding_sites"],
    site_boost=3.0,
)
print(f"Annotation-weighted shape: {emb_ann.shape}")

In [ ]:
# Strategy 3: Expression-context concatenation
emb_ctx = wpe.expression_context_embedding(
    "TP53", model="esm2_650M", use_gtex=True,
)
print(f"Expression-context shape: {emb_ctx.shape}")
print(f"  (protein embedding + GTEx tissue vector)")

In [ ]:
# All strategies at once
all_embs = wpe.embed_perturbation(
    "TP53", model="esm2_650M", strategy="full",
    tpm_values={"P04637": 90.0, "P04637-2": 10.0},
    use_gtex=True,
)
for name, emb in all_embs.items():
    print(f"  {name:25s}: shape {emb.shape}")

## Summary

| What | How |
|---|---|
| Gene pathways | ``GeneAnnotator().get_pathways(gene)`` |
| Tissue expression | ``GeneAnnotator().get_tissue_expression(gene)`` |
| PPI partners | ``GeneAnnotator().get_protein_interactions(gene)`` |
| Disease associations | ``GeneAnnotator().get_disease_associations(gene)`` |
| UniProt function | ``ProteinAnnotator().annotate(gene, sources='function')`` |
| Subcellular location | ``ProteinAnnotator().annotate(gene, sources='location')`` |
| Functional sites | ``ProteinAnnotator().annotate(gene, sources='sites')`` |
| GO terms | ``ProteinAnnotator().annotate(gene, sources='go')`` |
| TPM-weighted embedding | ``WeightedProteinEmbedder.tpm_weighted_embedding(gene, model, tpm_values)`` |
| Site-weighted pooling | ``WeightedProteinEmbedder.annotation_weighted_embedding(gene, model)`` |
| Expression context | ``WeightedProteinEmbedder.expression_context_embedding(gene, model, use_gtex=True)`` |